# 02 - Data Transformation

## Project
India Air Quality Intelligence — From Raw Data to Pollution Risk Insights

## Objective
Transform the original wide-format CPCB air quality dataset
into a structured hourly-level dataset suitable for analysis.

## Tasks
1. Load and inspect the raw dataset.
2. Convert wide-format data into long format.
3. Create timestamps and station identifiers.
4. Validate row counts and missing values.
5. Identify and handle data-quality issues.
6. Save the validated processed dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Define project directory
PROJECT_ROOT = Path.cwd().parent

# Define raw dataset path
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cpcb-aqi.csv"

# Verify dataset exists
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

# Load dataset
df = pd.read_csv(DATA_PATH)

# Basic inspection
print("\nDataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset exists: True
Dataset path: c:\Users\khush\Projects\india-air-quality-intelligence\data\raw\cpcb-aqi.csv

Dataset loaded successfully!
Shape: (562878, 29)

Columns:
['Station ID', 'State', 'City', 'Station Name', 'Date', '00:00:00', '01:00:00', '02:00:00', '03:00:00', '04:00:00', '05:00:00', '06:00:00', '07:00:00', '08:00:00', '09:00:00', '10:00:00', '11:00:00', '12:00:00', '13:00:00', '14:00:00', '15:00:00', '16:00:00', '17:00:00', '18:00:00', '19:00:00', '20:00:00', '21:00:00', '22:00:00', '23:00:00']


,Station ID,State,City,Station Name,Date,00:00:00,01:00:00,02:00:00,03:00:00,04:00:00,...,14:00:00,15:00:00,16:00:00,17:00:00,18:00:00,19:00:00,20:00:00,21:00:00,22:00:00,23:00:00
0,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-11,NaN,NaN,NaN,NaN,NaN,...,178.0,196.0,205.0,201.0,114.0,136.0,126.0,253.0,232.0,200.0
1,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-12,263.0,305.0,312.0,315.0,314.0,...,85.0,96.0,113.0,131.0,89.0,106.0,175.0,261.0,262.0,159.0
2,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-13,137.0,176.0,216.0,208.0,159.0,...,135.0,138.0,146.0,140.0,115.0,160.0,138.0,108.0,112.0,89.0
3,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-14,85.0,85.0,97.0,137.0,148.0,...,105.0,113.0,132.0,102.0,139.0,197.0,193.0,172.0,146.0,154.0
4,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-15,127.0,135.0,134.0,151.0,166.0,...,125.0,136.0,143.0,134.0,108.0,146.0,188.0,243.0,194.0,188.0


In [2]:
# Identify hourly columns
hourly_cols = [
    col for col in df.columns
    if col.endswith(":00:00")
]

# Validate hourly columns
print("Number of hourly columns:", len(hourly_cols))

print("\nHourly columns:")
print(hourly_cols)

print("\nOriginal dataset shape:", df.shape)

print("\nExpected transformed rows:")
print(len(df) * len(hourly_cols))

# Check date column
print("\nDate data type:", df["Date"].dtype)

# Check AQI data types
print("\nHourly column data types:")
print(df[hourly_cols].dtypes.value_counts())

Number of hourly columns: 24

Hourly columns:
['00:00:00', '01:00:00', '02:00:00', '03:00:00', '04:00:00', '05:00:00', '06:00:00', '07:00:00', '08:00:00', '09:00:00', '10:00:00', '11:00:00', '12:00:00', '13:00:00', '14:00:00', '15:00:00', '16:00:00', '17:00:00', '18:00:00', '19:00:00', '20:00:00', '21:00:00', '22:00:00', '23:00:00']

Original dataset shape: (562878, 29)

Expected transformed rows:
13509072

Date data type: str

Hourly column data types:
float64    24
Name: count, dtype: int64


In [3]:
# Convert Date to datetime
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

# Transform wide format into long format
df_long = df.melt(
    id_vars=[
        "Station ID",
        "State",
        "City",
        "Station Name",
        "Date"
    ],
    value_vars=hourly_cols,
    var_name="Hour",
    value_name="AQI"
)

# Convert AQI to numeric
df_long["AQI"] = pd.to_numeric(
    df_long["AQI"],
    errors="coerce"
)

# Create timestamp
df_long["Timestamp"] = pd.to_datetime(
    df_long["Date"].dt.strftime("%Y-%m-%d")
    + " "
    + df_long["Hour"],
    errors="coerce"
)

# Create a reliable station identity
df_long["Station"] = (
    df_long["State"].astype(str) + " | "
    + df_long["City"].astype(str) + " | "
    + df_long["Station Name"].astype(str)
)

print("Transformation completed!")

print("\nTransformed dataset shape:", df_long.shape)

print("\nMissing AQI readings:", df_long["AQI"].isna().sum())

print("Valid AQI readings:", df_long["AQI"].notna().sum())

print("\nFirst 10 rows:")
display(df_long.head(10))

Transformation completed!

Transformed dataset shape: (13509072, 9)

Missing AQI readings: 1035531
Valid AQI readings: 12473541

First 10 rows:


,Station ID,State,City,Station Name,Date,Hour,AQI,Timestamp,Station
0,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-11,00:00:00,NaN,2017-12-11,Andhra Pradesh | Amaravati | Secretariat Amara...
1,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-12,00:00:00,263.0,2017-12-12,Andhra Pradesh | Amaravati | Secretariat Amara...
2,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-13,00:00:00,137.0,2017-12-13,Andhra Pradesh | Amaravati | Secretariat Amara...
3,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-14,00:00:00,85.0,2017-12-14,Andhra Pradesh | Amaravati | Secretariat Amara...
4,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-15,00:00:00,127.0,2017-12-15,Andhra Pradesh | Amaravati | Secretariat Amara...
5,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-16,00:00:00,153.0,2017-12-16,Andhra Pradesh | Amaravati | Secretariat Amara...
6,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-17,00:00:00,280.0,2017-12-17,Andhra Pradesh | Amaravati | Secretariat Amara...
7,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-18,00:00:00,314.0,2017-12-18,Andhra Pradesh | Amaravati | Secretariat Amara...
8,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-19,00:00:00,256.0,2017-12-19,Andhra Pradesh | Amaravati | Secretariat Amara...
9,site_1406,Andhra Pradesh,Amaravati,Secretariat Amaravati APPCB,2017-12-20,00:00:00,294.0,2017-12-20,Andhra Pradesh | Amaravati | Secretariat Amara...


In [4]:
# 1. Check data types
print("Timestamp data type:", df_long["Timestamp"].dtype)
print("Date data type:", df_long["Date"].dtype)

# 2. Check missing timestamps
print("\nMissing timestamps:")
print(df_long["Timestamp"].isna().sum())

# 3. Check timestamp range
print("\nFirst timestamp:", df_long["Timestamp"].min())
print("Last timestamp:", df_long["Timestamp"].max())

# 4. Check whether all timestamps are at hourly intervals
print("\nTimestamp minutes distribution:")
print(df_long["Timestamp"].dt.minute.value_counts())

# 5. Check duplicate station-timestamp combinations
duplicate_count = df_long.duplicated(
    subset=["Station", "Timestamp"]
).sum()

print("\nDuplicate station-timestamp records:", duplicate_count)

# 6. Check that all 24 hours are represented
print("\nUnique hours:", df_long["Hour"].nunique())
print(sorted(df_long["Hour"].unique()))

# 7. Verify row-count consistency
assert len(df_long) == len(df) * 24
assert df_long["AQI"].notna().sum() == 12473541
assert df_long["AQI"].isna().sum() == 1035531

print("\nAll row-count and AQI preservation checks passed!")

Timestamp data type: datetime64[us]
Date data type: datetime64[us]

Missing timestamps:
0

First timestamp: 2017-01-01 00:00:00
Last timestamp: 2023-12-31 23:00:00

Timestamp minutes distribution:
Timestamp
0    13509072
Name: count, dtype: int64

Duplicate station-timestamp records: 0

Unique hours: 24
['00:00:00', '01:00:00', '02:00:00', '03:00:00', '04:00:00', '05:00:00', '06:00:00', '07:00:00', '08:00:00', '09:00:00', '10:00:00', '11:00:00', '12:00:00', '13:00:00', '14:00:00', '15:00:00', '16:00:00', '17:00:00', '18:00:00', '19:00:00', '20:00:00', '21:00:00', '22:00:00', '23:00:00']

All row-count and AQI preservation checks passed!


In [5]:
# Create a working copy
df_long["AQI_Status"] = "valid"

# Identify missing readings
df_long.loc[
    df_long["AQI"].isna(),
    "AQI_Status"
] = "missing"

# Identify readings above the standard AQI scale
df_long.loc[
    df_long["AQI"] > 500,
    "AQI_Status"
] = "out_of_range"

# Check quality flag distribution
print("AQI quality status distribution:")

display(
    df_long["AQI_Status"]
    .value_counts(dropna=False)
    .to_frame("Count")
)

# Calculate percentages
print("\nPercentage distribution:")

display(
    (
        df_long["AQI_Status"]
        .value_counts(normalize=True)
        .mul(100)
        .round(4)
    ).to_frame("Percentage")
)

# Inspect out-of-range readings
out_of_range = df_long[
    df_long["AQI_Status"] == "out_of_range"
]

print("\nOut-of-range readings:", len(out_of_range))

display(
    out_of_range[
        [
            "State",
            "City",
            "Station Name",
            "Timestamp",
            "AQI",
            "AQI_Status"
        ]
    ].head(15)
)

AQI quality status distribution:


,Count
AQI_Status,
valid,12473485
missing,1035531
out_of_range,56



Percentage distribution:


,Percentage
AQI_Status,
valid,92.3341
missing,7.6654
out_of_range,0.0004



Out-of-range readings: 56


,State,City,Station Name,Timestamp,AQI,AQI_Status
404624,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-21 00:00:00,900.0,out_of_range
404625,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-22 00:00:00,900.0,out_of_range
967502,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-21 01:00:00,900.0,out_of_range
967503,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-22 01:00:00,900.0,out_of_range
1118307,West Bengal,Kolkata,Rabindra Bharati University Kolkata WBPCB,2017-10-06 01:00:00,3562.0,out_of_range
1118318,West Bengal,Kolkata,Rabindra Bharati University Kolkata WBPCB,2017-10-17 01:00:00,2176.0,out_of_range
1118321,West Bengal,Kolkata,Rabindra Bharati University Kolkata WBPCB,2017-10-20 01:00:00,2164.0,out_of_range
1530380,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-21 02:00:00,900.0,out_of_range
1530381,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-22 02:00:00,900.0,out_of_range
2093258,Punjab,Ludhiana,Punjab Agricultural University Ludhiana PPCB,2017-09-21 03:00:00,900.0,out_of_range


In [6]:
# Create analytical copy
df_analytical = df_long.copy()

# Create a separate analysis-ready AQI column
df_analytical["AQI_Valid"] = df_analytical["AQI"].where(
    df_analytical["AQI_Status"] == "valid"
)

# Verify analytical dataset
print("Analytical dataset shape:", df_analytical.shape)

print("\nOriginal AQI non-missing readings:",
      df_analytical["AQI"].notna().sum())

print("Valid AQI readings for analysis:",
      df_analytical["AQI_Valid"].notna().sum())

print("Excluded out-of-range readings:",
      df_analytical["AQI_Status"].eq("out_of_range").sum())

print("\nAQI summary statistics:")

display(
    df_analytical["AQI_Valid"].describe().round(2)
)

# Verify original values remain unchanged
print("\nOriginal AQI column preserved:",
      df_analytical["AQI"].equals(df_long["AQI"]))

Analytical dataset shape: (13509072, 11)

Original AQI non-missing readings: 12473541
Valid AQI readings for analysis: 12473485
Excluded out-of-range readings: 56

AQI summary statistics:


count    12473485.00
mean          136.46
std           107.56
min             1.00
25%            60.00
50%           100.00
75%           173.00
max           500.00
Name: AQI_Valid, dtype: float64


Original AQI column preserved: True


In [7]:
# Check Parquet engine
try:
    import pyarrow
    print("PyArrow version:", pyarrow.__version__)
    print("Parquet support is ready!")

except ImportError:
    print("PyArrow is not installed.")

PyArrow version: 26.0.0
Parquet support is ready!


In [8]:
# Create processed data directory
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Define output paths
HOURLY_PATH = PROCESSED_DIR / "air_quality_hourly.parquet"
ANALYTICAL_PATH = PROCESSED_DIR / "air_quality_analytical.parquet"

# Save complete transformed dataset
df_long.to_parquet(
    HOURLY_PATH,
    engine="pyarrow",
    compression="snappy",
    index=False
)

# Save analytical dataset
df_analytical.to_parquet(
    ANALYTICAL_PATH,
    engine="pyarrow",
    compression="snappy",
    index=False
)

print("Datasets saved successfully!")

print("\nComplete dataset:", HOURLY_PATH)
print("Analytical dataset:", ANALYTICAL_PATH)

print("\nFile sizes:")
print("Complete:", round(HOURLY_PATH.stat().st_size / (1024**2), 2), "MB")
print("Analytical:", round(ANALYTICAL_PATH.stat().st_size / (1024**2), 2), "MB")

Datasets saved successfully!

Complete dataset: c:\Users\khush\Projects\india-air-quality-intelligence\data\processed\air_quality_hourly.parquet
Analytical dataset: c:\Users\khush\Projects\india-air-quality-intelligence\data\processed\air_quality_analytical.parquet

File sizes:
Complete: 45.47 MB
Analytical: 60.17 MB


In [9]:
# Reload saved analytical dataset
df_check = pd.read_parquet(ANALYTICAL_PATH)

print("Reloaded shape:", df_check.shape)
print("Original shape:", df_analytical.shape)

print("\nAQI missing values:", df_check["AQI"].isna().sum())
print("Valid AQI readings:", df_check["AQI_Valid"].notna().sum())

print("\nTimestamp type:", df_check["Timestamp"].dtype)

assert df_check.shape == df_analytical.shape
assert df_check["AQI_Valid"].notna().sum() == 12473485

print("\nParquet verification successful!")

Reloaded shape: (13509072, 11)
Original shape: (13509072, 11)

AQI missing values: 1035531
Valid AQI readings: 12473485

Timestamp type: datetime64[us]

Parquet verification successful!


## Final Outcome

The original CPCB dataset was successfully transformed from wide format
into hourly long format.

### Outputs
- `air_quality_hourly.parquet`
- `air_quality_analytical.parquet`

### Validation
- Row counts verified.
- Timestamps successfully generated.
- Duplicate station-timestamp combinations checked.
- Missing values preserved.
- Out-of-range AQI values flagged.
- Parquet files successfully saved and reloaded.

### Key Decision
The `AQI_Valid` column will be used for standard AQI analysis.
The original `AQI` column is retained for data auditing.

**Status: Data transformation completed successfully.**